## RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_42796\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
a:\Teach me RAG\dRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf inside the directory

def process_all_pdfs(pdf_directory):
    "Process all pdf file on directory"
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error : {e}")
    
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

In [3]:
all_pdf_documents = process_all_pdfs("../data/NepalConstitution2015")

Found 10 PDF files to process
Processing: Preamble.pdf
Loaded 1 pages
Processing: Article 1 - Constitution as the fundamental law.pdf
Loaded 1 pages
Processing: Article 2 - Sovereignty and state authority.pdf
Loaded 1 pages
Processing: Article 3 - Nation.pdf
Loaded 1 pages
Processing: Article 4 - State of Nepal.pdf
Loaded 1 pages
Processing: Article 5 - National interest.pdf
Loaded 1 pages
Processing: Article 6 - Languages of the nation.pdf
Loaded 1 pages
Processing: Article 7 - Official language.pdf
Loaded 1 pages
Processing: Article 8 - National flag.pdf
Loaded 1 pages
Processing: Article 9 - National Anthem, etc.pdf
Loaded 1 pages
Total documents loaded: 10


In [4]:
## Text Spliting into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    "Split documents into smaller chunks for better RAG performance"
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [5]:
chunks = split_documents(all_pdf_documents)

Split 10 documents into 12 chunks
Example chunk:
Content: We, the Sovereign People of Nepal, 
Internalizing the people’s sovereign right and right to autonomy and self-
rule, while maintaining freedom, sovereignty, territorial integrity, national 
unity, ind...
Metadata: {'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-06-20T14:26:13+05:45', 'author': 'Dip Neupane', 'moddate': '2026-06-20T14:26:13+05:45', 'source': '..\\data\\NepalConstitution2015\\Preamble.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Preamble.pdf', 'file_type': 'pdf'}


### embedding and vector store db


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    "Handles document embedding generation using sentence transformer"

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initializing the embedding manager

        Args:
            model_name: Huggingface model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not found")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## Initialize the embedding manager
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10787.12it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_42796\3161585508.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


###  Vector Store

In [18]:
class VectorStore:
    """manages document embeddings in a ChormaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        initialize the vector store

        Args:
            collection_name: Name of the ChormaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initilize_store()

    def _initilize_store(self):
        """Initilize ChormsDB client and collection"""

        try:
            # Create persistent ChormsDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialize. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of langchain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            #Enmedding
            embeddings_list.append(embedding.tolist())

            # Add to collection
            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )
                print(f"Successfully added {len(documents)} documents to vector store")
                print(f"Total documents in collection: {self.collection.count()}")

            except Exception as e:
                print(f"Error adding documents to the vector store: {e}")
                raise

vectorStore = VectorStore()

Vector store initialize. Collection: pdf_documents
Existing documents in collection: 0


In [19]:
##Convert the text into embeddings
texts = [doc.page_content for doc in chunks]

## Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database
vectorStore.add_documents(chunks, embeddings)
 

Generating embeddings for 12 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.47it/s]


Generated embeddings with shape: (12, 384)
Adding 12 documents to vector store...
Successfully added 12 documents to vector store
Total documents in collection: 1
Successfully added 12 documents to vector store
Total documents in collection: 2
Successfully added 12 documents to vector store
Total documents in collection: 3
Successfully added 12 documents to vector store
Total documents in collection: 4
Successfully added 12 documents to vector store
Total documents in collection: 5
Successfully added 12 documents to vector store
Total documents in collection: 6
Successfully added 12 documents to vector store
Total documents in collection: 7
Successfully added 12 documents to vector store
Total documents in collection: 8
Successfully added 12 documents to vector store
Total documents in collection: 9
Successfully added 12 documents to vector store
Total documents in collection: 10
Successfully added 12 documents to vector store
Total documents in collection: 11
Successfully added 12 doc